# 02 — Quiz Bot: Interactive MCQ Quiz powered by Groq

### Core concepts covered
| Concept | Description |
|---------|-------------|
| Conditional Edge | Route to different nodes based on a function |
| Loop | Conditional edge that circles back to an earlier node |
| `add_messages` | State reducer that appends rather than overwrites |
| Groq LLM | Free LLM API — generates questions dynamically |

### Graph flow
```
START → generate_question → get_answer ──(total < 3)──▶ generate_question
                                        ──(total >= 3)─▶ show_result → END
```

> **Before running:** set `GROQ_API_KEY` in a `.env` file in this folder.

## 1. Install Dependencies

In [ ]:
# !pip install langgraph langchain langchain-groq python-dotenv

## 2. Imports & Setup

In [ ]:
import os
from typing import TypedDict, Annotated
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

load_dotenv()
print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))

## 3. Initialise the LLM

In [ ]:
llm = init_chat_model("groq:llama3-8b-8192")

MAX_QUESTIONS = 3   # change to make the quiz longer or shorter

## 4. Define the State

`messages` uses the `add_messages` reducer — instead of replacing the list,
it **appends** new messages. This keeps the full conversation history.

In [ ]:
class QuizState(TypedDict):
    topic:            str
    score:            int
    total:            int
    current_question: str
    correct_answer:   str
    messages:         Annotated[list, add_messages]

## 5. Define the Nodes

In [ ]:
def generate_question(state: QuizState) -> dict:
    """Ask Groq to generate the next MCQ for the given topic."""
    q_num  = state["total"] + 1
    prompt = (
        f'Create a single multiple-choice question about "{state["topic"]}".\n\n'
        f'Use EXACTLY this format (no extra text):\n\n'
        f'Question {q_num}: <your question>\n'
        f'A) <option>\nB) <option>\nC) <option>\nD) <option>\n'
        f'Correct: <single letter A, B, C, or D>\n\n'
        f'Keep it clear and concise.'
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    content  = response.content.strip()

    # Split off the answer line so it is never shown to the user
    lines       = content.split("\n")
    answer_line = next((l for l in lines if l.lower().startswith("correct:")), "Correct: A")
    correct     = answer_line.split(":", 1)[1].strip().upper()[0]
    question_text = "\n".join(l for l in lines if not l.lower().startswith("correct:"))

    print(f"\n{question_text}\n")
    return {
        "current_question": question_text,
        "correct_answer":   correct,
        "messages":         [AIMessage(content=question_text)],
    }

In [ ]:
def get_answer(state: QuizState) -> dict:
    """Prompt the user, check the answer, update score."""
    user_input = input("Your answer (A / B / C / D): ").strip().upper()

    if user_input not in ("A", "B", "C", "D"):
        print("  Invalid input — defaulting to A.")
        user_input = "A"

    correct    = state["correct_answer"]
    is_correct = user_input == correct
    score      = state["score"] + (1 if is_correct else 0)
    total      = state["total"] + 1
    feedback   = "  ✅ Correct!" if is_correct else f"  ❌ Wrong! The correct answer was {correct}."

    print(feedback)
    return {
        "score":    score,
        "total":    total,
        "messages": [HumanMessage(content=user_input), AIMessage(content=feedback)],
    }

In [ ]:
def show_result(state: QuizState) -> dict:
    """Display the final score with a motivational message."""
    score = state["score"]
    total = state["total"]
    pct   = (score / total) * 100

    if pct == 100:   badge = "Perfect score! Outstanding! 🏆"
    elif pct >= 66:  badge = "Great job! Keep it up! 🎉"
    elif pct >= 33:  badge = "Good try! Keep practicing. 📚"
    else:            badge = "Keep studying — you'll improve! 💪"

    result = (
        f"\n{'═'*38}\n"
        f"  Quiz Complete on: {state['topic']}\n"
        f"  Score : {score}/{total}  ({pct:.0f}%)\n"
        f"  {badge}\n"
        f"{'═'*38}"
    )
    print(result)
    return {"messages": [AIMessage(content=result)]}

## 6. Conditional Edge — The Loop

This function decides where to go after every answer:
- If `total < MAX_QUESTIONS` → loop back to `generate_question`
- If `total >= MAX_QUESTIONS` → go to `show_result`

In [ ]:
def should_continue(state: QuizState) -> str:
    """Route: loop for another question, or stop when done."""
    return "show_result" if state["total"] >= MAX_QUESTIONS else "generate_question"

## 7. Build & Compile the Graph

In [ ]:
builder = StateGraph(QuizState)

builder.add_node("generate_question", generate_question)
builder.add_node("get_answer",        get_answer)
builder.add_node("show_result",       show_result)

builder.add_edge(START, "generate_question")
builder.add_edge("generate_question", "get_answer")

# Conditional edge — the loop lives here
builder.add_conditional_edges(
    "get_answer",
    should_continue,
    {"generate_question": "generate_question", "show_result": "show_result"},
)
builder.add_edge("show_result", END)

graph = builder.compile()
print("Graph compiled!")

## 8. Visualise the Graph *(optional)*

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Visualisation skipped: {e}")

## 9. Run the Quiz

Type a topic and answer each question with **A**, **B**, **C**, or **D**.

In [ ]:
topic = input("Enter a quiz topic (e.g., Python, Space, World History): ").strip()
if not topic:
    topic = "General Knowledge"

print(f"\nStarting {MAX_QUESTIONS}-question quiz on: {topic}")
print("─" * 40)

graph.invoke({
    "topic":            topic,
    "score":            0,
    "total":            0,
    "current_question": "",
    "correct_answer":   "",
    "messages":         [],
})